In [3]:
# coding: utf-8
import sys
sys.path.append('..')
import numpy as np
from common.layers import MatMul, SoftmaxWithLoss


class SimpleCBOW:
    def __init__(self, vocab_size, hidden_size):
        # V:語彙数　H:ベクトルの次元数
        V, H = vocab_size, hidden_size

        # 重みの初期化
        W_in = 0.01 * np.random.randn(V, H).astype('f')   # 単語（one-hot）をベクトルに変える（第4章のEmbedding層）
        W_out = 0.01 * np.random.randn(H, V).astype('f')  # ベクトルを単語に戻す

        # レイヤの生成
        self.in_layer0 = MatMul(W_in)              # 第4章ではMatMulレイヤからEmbeddingレイヤに差し変わる
        self.in_layer1 = MatMul(W_in)             # 入力層が2つあるのは、ターゲットの前後の単語を同時に処理したいから
        self.out_layer = MatMul(W_out)
        self.loss_layer = SoftmaxWithLoss()  # 第4章ではNegative Samplingに変わる

        # すべての重みと勾配をリストにまとめる
        layers = [self.in_layer0, self.in_layer1, self.out_layer]
        self.params, self.grads = [], []  # 重み（params）とズレ（grads）それぞれリストを作成
        for layer in layers:  # layersにまとめたレイヤを1つ1つ取り出して、
            self.params += layer.params  # 重みはparamsに、
            self.grads += layer.grads    # ズレはgradsにそれぞれ代入

        # メンバ変数に単語の分散表現を設定
        self.word_vecs = W_in  # W_inを入れるのは、単語ベクトルを保持し続けてくれるから、使いまわしやすい

    def forward(self, contexts, target):
        h0 = self.in_layer0.forward(contexts[:, 0])  # 前の単語を行列のかけ算（MatMul）に通す
        h1 = self.in_layer1.forward(contexts[:, 1])  # 後ろの単語を行列かけ算に通す
        h = (h0 + h1) * 0.5  # h0とh1の情報を混ぜて2で割る（推測するための共通のヒントを作るため）
        score = self.out_layer.forward(h)  # 合体したヒントhに、もう1枚の重みW_outをかける
        loss = self.loss_layer.forward(score, target) # スコアと正解を比べて、どれくらい外れたかの数値を出す
        return loss

    def backward(self, dout=1):
        ds = self.loss_layer.backward(dout)  # dsに順伝播で出たズレを渡す
        da = self.out_layer.backward(ds)
        da *= 0.5
        self.in_layer1.backward(da)
        self.in_layer0.backward(da)
        return None

In [ ]:
# coding: utf-8
import sys
sys.path.append('..')
import numpy as np
from common.layers import MatMul, SoftmaxWithLoss


class SimpleSkipGram:
    def __init__(self, vocab_size, hidden_size):
        V, H = vocab_size, hidden_size

        # 重みの初期化
        W_in = 0.01 * np.random.randn(V, H).astype('f')
        W_out = 0.01 * np.random.randn(H, V).astype('f')

        # レイヤの生成
        self.in_layer = MatMul(W_in)
        self.out_layer = MatMul(W_out)
        self.loss_layer1 = SoftmaxWithLoss()
        self.loss_layer2 = SoftmaxWithLoss()

        # すべての重みと勾配をリストにまとめる
        layers = [self.in_layer, self.out_layer]
        self.params, self.grads = [], []
        for layer in layers:
            self.params += layer.params
            self.grads += layer.grads

        # メンバ変数に単語の分散表現を設定
        self.word_vecs = W_in

    def forward(self, contexts, target):   
        h = self.in_layer.forward(target)   # 入力されたターゲットに重みを掛ける
        s = self.out_layer.forward(h)       # hに出口の重みを掛けて、全単語分のスコアを出す
        l1 = self.loss_layer1.forward(s, contexts[:, 0])  # 求めたスコアを使って、左隣の答え合わせ
        l2 = self.loss_layer2.forward(s, contexts[:, 1])
        loss = l1 + l2
        return loss

    def backward(self, dout=1):
        dl1 = self.loss_layer1.backward(dout)
        dl2 = self.loss_layer2.backward(dout)
        ds = dl1 + dl2
        dh = self.out_layer.backward(ds)
        self.in_layer.backward(dh)
        return None